# 🌫️ 02 — Dhara: Ingest Air Quality Data (Bronze)

Air Quality data from data.gov.in. **This is our fog-delay correlation signal** — the track spec explicitly calls out fog-delay correlation as a key feature. PM2.5 and NO2 levels spike during winter fog across North Indian stations and correlate strongly with morning train delays.

Pivots the long-format pollutant data into a wide station-level view suitable for joining with train runs.

In [0]:
%sql
USE CATALOG setu_rail;
USE SCHEMA bronze;

## Step 1 — Find the air quality CSV dynamically

In [0]:
# The AQ file has a UUID-style name. Find any CSV that isn't a train file.
files = dbutils.fs.ls("/Volumes/setu_rail/bronze/raw_files")
aq_candidates = [f.path for f in files
                 if f.name.endswith(".csv")
                 and not f.name.startswith("train_")
                 and not f.name.startswith("railways_")]

if not aq_candidates:
    raise FileNotFoundError("No air-quality CSV found. Expected a data.gov.in pollutant file.")

aq_path = aq_candidates[0]
print(f"✅ Using AQ file: {aq_path}")

## Step 2 — Raw Bronze load

In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date, upper, trim, when

aq_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(aq_path))

print(f"Rows: {aq_raw.count():,}")
aq_raw.printSchema()
aq_raw.show(5, truncate=False)

In [0]:
# Write raw Bronze snapshot
(aq_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.bronze.air_quality_raw"))

print("✅ bronze.air_quality_raw written")

## Step 3 — Pivot to wide format: one row per (city, date) with columns per pollutant

This is what ML training needs. The raw format is long (one row per pollutant per station per timestamp).

In [0]:
aq_clean = (spark.table("setu_rail.bronze.air_quality_raw")
    .withColumn("city_u", upper(trim(col("city"))))
    .withColumn("obs_ts",  to_timestamp(col("last_update"), "dd-MM-yyyy HH:mm:ss"))
    .withColumn("obs_date", to_date(col("obs_ts")))
    .withColumn("pollutant_id", trim(col("pollutant_id")))
    # Some pollutant_avg fields are "NA" strings — coerce
    .withColumn("pollutant_avg_num",
        when(col("pollutant_avg").cast("double").isNotNull(),
             col("pollutant_avg").cast("double")))
)

# Pivot: rows = (city, date), columns = pollutant types
aq_wide = (aq_clean
    .groupBy("city_u", "obs_date", "latitude", "longitude")
    .pivot("pollutant_id", ["PM2.5", "PM10", "NO2", "SO2", "CO", "OZONE", "NH3"])
    .avg("pollutant_avg_num")
    .withColumnRenamed("PM2.5", "pm25")
    .withColumnRenamed("PM10",  "pm10")
    .withColumnRenamed("NO2",   "no2")
    .withColumnRenamed("SO2",   "so2")
    .withColumnRenamed("CO",    "co")
    .withColumnRenamed("OZONE", "ozone")
    .withColumnRenamed("NH3",   "nh3")
    .withColumnRenamed("city_u", "city")
)

print(f"Pivoted rows: {aq_wide.count():,}")
aq_wide.show(5, truncate=False)

In [0]:
(aq_wide.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("setu_rail.bronze.air_quality"))

print("✅ bronze.air_quality (pivoted) written")

## Step 4 — Validate + show fog-correlation-ready data

In [0]:
%sql
-- Top-polluted cities — these are the stations most likely to have fog-induced delays
SELECT city,
       ROUND(AVG(pm25), 1) AS avg_pm25,
       ROUND(AVG(no2), 1)  AS avg_no2,
       COUNT(*)            AS observation_days
FROM   setu_rail.bronze.air_quality
WHERE  pm25 IS NOT NULL
GROUP BY city
ORDER BY avg_pm25 DESC
LIMIT 15;

✅ **Next:** `03_ingest_rulebook_pdfs.ipynb`